# Bronze Layer - Import from Unity Catalog Volume

**Source Volume:** `/Volumes/workspace/rebu/raw`  
**Target Tables:** `workspace.rebu.bronze_{table}`

This notebook reads CSV files from a Unity Catalog Volume and creates Bronze layer tables.

---

**Notebook Summary:**
- Configures Unity Catalog and volume paths for data ingestion.
- Verifies access to the source volume and lists available CSV files.
- Defines a helper function to ingest CSV files and add metadata columns.
- Ingests dimension tables (Taxi Type, Taxi, Drivers, Passengers, Payment Type, Place) into bronze tables.
- Prepares for ingestion of fact tables.

## Configuration

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col

# Unity Catalog Configuration
UC_CATALOG = "workspace"
UC_SCHEMA = "rebu"
VOLUME_NAME = "raw"

# Volume path where CSV files are stored
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{VOLUME_NAME}"

print("="*80)
print("BRONZE LAYER CONFIGURATION")
print("="*80)
print(f"Source Volume Path: {VOLUME_PATH}")
print(f"Target Catalog: {UC_CATALOG}")
print(f"Target Schema: {UC_SCHEMA}")
print(f"Bronze Tables: {UC_CATALOG}.{UC_SCHEMA}.bronze_*")
print("="*80)

## Verify Volume Access

In [0]:
# List all files in the volume
print(f"\nFiles in {VOLUME_PATH}:")
print("-"*80)

try:
    files = dbutils.fs.ls(VOLUME_PATH)
    csv_files = [f for f in files if f.name.endswith('.csv')]
    
    if csv_files:
        for file in csv_files:
            size_mb = file.size / (1024 * 1024)
            print(f"{file.name:<40} {size_mb:>10.2f} MB")
        print(f"\n✓ Found {len(csv_files)} CSV file(s)")
    else:
        print("No CSV files found in volume")
        
except Exception as e:
    print(f"  ✗ Error accessing volume: {e}")
    print("\n  Troubleshooting:")
    print("  1. Check volume exists: SHOW VOLUMES IN workspace.rebu")
    print("  2. Check permissions: GRANT READ FILES ON VOLUME workspace.rebu.raw TO `user@email.com`")
    print("  3. Verify path: /Volumes/workspace/rebu/raw")

print("-"*80)

## Helper Function

In [0]:
def ingest_csv_to_bronze(table_name, csv_file_name):
    """
    Read CSV from Unity Catalog Volume and create Bronze table
    
    Args:
        table_name: Name for the bronze table (e.g., "taxi")
        csv_file_name: CSV filename in the volume (e.g., "Taxi.csv")
    
    Returns:
        DataFrame with metadata columns added
    """
    print(f"\n{'='*80}")
    print(f"Ingesting: {csv_file_name}")
    print(f"{'='*80}")
    
    # Full path to CSV file in Volume
    csv_path = f"{VOLUME_PATH}/{csv_file_name}"
    print(f"Source: {csv_path}")
    
    try:
        # Read CSV from Volume
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(csv_path)
        
        # Show schema
        print(f"\nSchema:")
        df.printSchema()
        
        # Add Bronze layer metadata
        df_with_metadata = df \
            .withColumn("ingestion_timestamp", current_timestamp()) \
            .withColumn("source_file", lit(csv_file_name)) \
            .withColumn("source_volume", lit(VOLUME_PATH))
        
        # Create Bronze table name
        bronze_table_name = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_{table_name}"
        
        # Save as Unity Catalog managed table
        print(f"\nTarget: {bronze_table_name}")
        
        df_with_metadata.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(bronze_table_name)
        
        record_count = df_with_metadata.count()
        print(f"\nIngested {record_count:,} records")
        print(f"{'='*80}\n")
        
        return df_with_metadata
        
    except Exception as e:
        print(f"\nERROR: {str(e)}")
        print(f"{'='*80}\n")
        raise

## Ingest Dimension Tables

### Taxi Type

In [0]:
df_taxi_type = ingest_csv_to_bronze("taxi_type", "TaxiType.csv")
display(df_taxi_type)

### Taxi

In [0]:
df_taxi = ingest_csv_to_bronze("taxi", "Taxi.csv")
display(df_taxi.limit(10))

### Drivers

In [0]:
df_drivers = ingest_csv_to_bronze("drivers", "Drivers.csv")
display(df_drivers.limit(10))

### Passengers

In [0]:
df_passengers = ingest_csv_to_bronze("passengers", "Passengers.csv")
display(df_passengers.limit(10))

### Payment Type

In [0]:
df_payment_type = ingest_csv_to_bronze("payment_type", "PaymentType.csv")
display(df_payment_type)

### Place (Location)

In [0]:
df_place = ingest_csv_to_bronze("place", "Place.csv")
display(df_place.limit(10))

## Ingest Fact Tables

### Booking

In [0]:
df_booking = ingest_csv_to_bronze("booking", "Booking.csv")
display(df_booking.limit(10))

### Rewards

In [0]:
df_rewards = ingest_csv_to_bronze("rewards", "Rewards.csv")
display(df_rewards.limit(10))

## Data Quality Summary

In [0]:
# Get table statistics
bronze_tables = [
    "taxi_type", "taxi", "drivers", "passengers", 
    "payment_type", "place", "booking", "rewards"
]

print("="*80)
print("BRONZE LAYER - DATA INGESTION SUMMARY")
print("="*80)
print(f"\nSource: {VOLUME_PATH}")
print(f"Target: {UC_CATALOG}.{UC_SCHEMA}\n")

total_records = 0
for table in bronze_tables:
    table_name = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_{table}"
    try:
        count = spark.table(table_name).count()
        print(f"  {table_name:<50} {count:>10,} records")
        total_records += count
    except Exception as e:
        print(f"  {table_name:<50} {'ERROR':>10}")

print("-"*80)
print(f"  {'TOTAL':<50} {total_records:>10,} records")
print("="*80)

## Verify Bronze Tables

In [0]:
# Show all Bronze tables in schema
print(f"\nBronze Tables in {UC_CATALOG}.{UC_SCHEMA}:")
print("-"*80)

spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{UC_SCHEMA}") \
    .filter("tableName LIKE 'bronze_%'") \
    .select("database", "tableName", "isTemporary") \
    .show(truncate=False)

## Sample Data Check

In [0]:
# Quick sample from each table
print("\nSample Data from Bronze Tables:")
print("="*80)

for table in ["taxi_type", "taxi", "drivers"][:3]:  # Show first 3
    table_name = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_{table}"
    print(f"\n{table_name}:")
    print("-"*80)
    spark.table(table_name).show(3, truncate=False)

## Table Metadata

In [0]:
# Show detailed info for one table
print(f"\nTable Details: {UC_CATALOG}.{UC_SCHEMA}.bronze_taxi_type")
print("="*80)
spark.sql(f"DESCRIBE EXTENDED {UC_CATALOG}.{UC_SCHEMA}.bronze_taxi_type").show(50, False)

## End of Bronze Layer Notebook

**Summary:**
- Read CSV files from Unity Catalog Volume
- Created 8 Bronze layer tables
- Added metadata columns for lineage
- All tables stored as Unity Catalog managed tables

**Next Steps:**
1. Run Silver Layer notebook for data cleaning
2. Update Silver layer to read from: `workspace.rebu.bronze_{table}`
3. Continue with Gold layer and OBT notebooks

## Troubleshooting Commands

In [0]:
# Run if you need to troubleshoot

# # Check volume exists
spark.sql("SHOW VOLUMES IN workspace.rebu").show()



In [0]:
# # Check table exists
spark.sql("SHOW TABLES IN workspace.rebu").show()



In [0]:
# # Check permissions
spark.sql("SHOW GRANTS ON VOLUME workspace.rebu.raw").show()



In [0]:
# # Re-read a specific file
test_df = spark.read.csv("/Volumes/workspace/rebu/raw/TaxiType.csv", header=True)
display(test_df)



In [0]:
# # Drop and recreate a table if needed
# # spark.sql("DROP TABLE IF EXISTS workspace.rebu.bronze_taxi_type")
# # df_taxi_type = ingest_csv_to_bronze("taxi_type", "TaxiType.csv")